<div align="center">
    <span style="font-size:30px">
        <strong>
            <!-- Símbolo de Python -->
            <img
                src="https://cdn3.emoji.gg/emojis/1887_python.png"
                style="margin-bottom:-5px"
                width="30px" 
                height="30px"
            >
            <!-- Título -->
            Python para Geólogos
            <!-- Versión -->
            <img 
                src="https://img.shields.io/github/release/kevinalexandr19/manual-python-geologia.svg?style=flat&label=&color=blue"
                style="margin-bottom:-2px" 
                width="40px"
            >
        </strong>
    </span>
    <br>
    <span>
        <!-- Github del proyecto -->
        <a href="https://github.com/kevinalexandr19/manual-python-geologia" target="_blank">
            <img src="https://img.shields.io/github/stars/kevinalexandr19/manual-python-geologia.svg?style=social&label=Github Repo">
        </a>
        &nbsp;&nbsp;
        <!-- Licencia -->
        <img src="https://img.shields.io/github/license/kevinalexandr19/manual-python-geologia.svg?color=forestgreen">
        &nbsp;&nbsp;
        <!-- Release date -->
        <img src="https://img.shields.io/github/release-date/kevinalexandr19/manual-python-geologia?color=gold">
    </span>
    <br>
    <span>
        <!-- Perfil de LinkedIn -->
        <a target="_blank" href="https://www.linkedin.com/in/kevin-alexander-gomez/">
            <img src="https://img.shields.io/badge/-Kevin Alexander Gomez-5eba00?style=social&logo=linkedin">
        </a>
        &nbsp;&nbsp;
        <!-- Perfil de Github -->
        <a target="_blank" href="https://github.com/kevinalexandr19">
            <img src="https://img.shields.io/github/followers/kevinalexandr19.svg?style=social&label=kevinalexandr19&maxAge=2592000">
        </a>
    </span>
    <br>
</div>

***

<span style="color:lightgreen; font-size:25px">**PG201 - Aprendizaje supervisado**</span>

Bienvenido al curso!!!

Vamos a revisar diferentes algoritmos de <span style="color:gold">aprendizaje supervisado</span> y su aplicación en Geología. <br>
Es necesario que tengas un conocimiento previo en programación con Python, álgebra lineal, estadística y geología.


<span style="color:gold; font-size:20px">**Extreme Gradient Boosting (XGBoost)** </span>

***
- [¿Qué es XGBoost?](#parte-1)
- [¿Cómo funciona XGBoost?](#parte-2)
- [Ejemplo de XGBoost con un skarn](#parte-3)
- [En conclusión...](#parte-4)

***

In [ ]:
import sys

if sys.platform == "emscripten":
    import piplite
    await piplite.install(['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'xgboost'])

<a id="parte-1"></a>

### <span style="color:lightgreen">**¿Qué es XGBoost?**</span>
***

XGBoost es una implementación de **gradient boosting** descrita por [Chen y Guestrin](https://arxiv.org/abs/1603.02754). Usa árboles pequeños, una tasa de aprendizaje y regularización para controlar cuánto aporta cada ajuste.

En un bosque aleatorio entrenamos árboles con distintas muestras y promediamos sus salidas. En boosting, cada ronda añade árboles al modelo que ya tenemos. La salida de una ronda cambia lo que tratará de ajustar la siguiente.

---

<a id="parte-2"></a>

### <span style="color:lightgreen">**¿Cómo funciona XGBoost?**</span>
***

1. Partimos de un puntaje base para cada clase.
2. Comparamos los puntajes actuales con las etiquetas mediante una función de pérdida.
3. Usamos las derivadas de la pérdida para elegir divisiones y valores de hoja para los nuevos árboles.
4. Sumamos a los puntajes el aporte de la ronda, escalado por la tasa de aprendizaje.

En regresión con pérdida cuadrática podemos pensar en los residuos. En esta clasificación, las correcciones vienen de las derivadas de la pérdida multiclase. Cada ronda puede añadir un árbol por clase. La pérdida de validación nos ayuda a decidir cuándo parar.

---

<a id="parte-3"></a>

### <span style="color:lightgreen">**Ejemplo de XGBoost con un skarn**</span>
***

Usaremos los segmentos del skarn y las dos vistas originales, una con nícoles paralelos (NP) y otra con nícoles cruzados (NC). El objetivo es reproducir tres anotaciones del conjunto de entrenamiento: `granate`, `oxidos` y `verde`. La última es una etiqueta visual y requiere evidencia petrográfica para asignarla a un mineral.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import xgboost as xgb
from sklearn.metrics import ConfusionMatrixDisplay

Para el navegador usamos una tabla pequeña extraída de las imágenes originales. Cada fila conserva el identificador de imagen y segmento, las coordenadas locales y seis valores RGB. En cada par de segmentos se comprobaron las dimensiones y la coincidencia del canal alfa. Luego se eligieron hasta 384 coordenadas opacas por segmento con semilla 42 y se leyeron ambas vistas en esas mismas coordenadas. Estas comprobaciones verifican la correspondencia de la máscara, y la interpretación depende también del registro de la captura.

In [ ]:
data = pd.read_csv("files/xgboost/skarn_pixeles.csv", dtype={"segmento": str})
columnas = ["R1", "G1", "B1", "R2", "G2", "B2"]
clases, y = np.unique(data["etiqueta"], return_inverse=True)
X = data[columnas].to_numpy()
data.head()

Toma una fila y lee sus seis entradas. Las tres primeras pertenecen a NP y las otras tres pertenecen a NC. La etiqueta de toda la fila viene de la anotación del segmento.

In [ ]:
print(dict(enumerate(clases)))
print(data.groupby("etiqueta")["segmento"].nunique())

Solo tenemos etiquetas de la imagen `skarn_1`. Reservamos tres segmentos completos, uno de cada clase, para validación. Distintos segmentos pueden proceder de zonas cercanas o solaparse en la foto original. Esta división mide comportamiento entre segmentos de la misma imagen.

In [ ]:
segmentos_val = ["8", "s0_9", "s6_17"]
validacion = data["segmento"].isin(segmentos_val).to_numpy()
ajuste = ~validacion
print(f"Filas de ajuste: {ajuste.sum():,} | Validación: {validacion.sum():,}")

XGBoost recibe la tabla numérica en un objeto `DMatrix`. Pasamos la etiqueta solo para ajustar o evaluar. Las columnas mantienen el mismo orden en entrenamiento, validación y predicción.

In [ ]:
dtrain = xgb.DMatrix(X[ajuste], label=y[ajuste], feature_names=columnas)
dval = xgb.DMatrix(X[validacion], label=y[validacion], feature_names=columnas)

Usaremos pérdida logarítmica multiclase. La parada temprana conserva la ronda con menor pérdida de validación. La curva y la matriz que siguen utilizan esos segmentos y sirven para revisar el ajuste.

In [ ]:
parametros = {
    "objective": "multi:softprob",
    "num_class": len(clases),
    "eval_metric": "mlogloss",
    "max_depth": 3,
    "eta": 0.15,
    "tree_method": "hist",
    "nthread": 1,
    "seed": 42,
}

In [ ]:
historial = {}
model = xgb.train(
    parametros, dtrain, num_boost_round=75,
    evals=[(dtrain, "ajuste"), (dval, "validación")],
    early_stopping_rounds=8, evals_result=historial, verbose_eval=False,
)
rondas = model.best_iteration + 1
print(f"Rondas elegidas por validación: {rondas}")

In [ ]:
proba_val = model.predict(dval, iteration_range=(0, rondas))
pred_val = proba_val.argmax(axis=1)
base = np.bincount(y[ajuste]).argmax()
print(f"Exactitud de referencia en validación: {(y[validacion] == base).mean():.1%}")
print(dict(zip(clases, proba_val[0].round(3))))

In [ ]:
fig, (ax, cm) = plt.subplots(1, 2, figsize=(10, 4))
for nombre, metricas in historial.items():
    perdida = metricas["mlogloss"]
    ax.plot(range(1, len(perdida) + 1), perdida, label=nombre)
ax.axvline(rondas, color="0.3", ls="--", label="ronda elegida")
ax.set(xlabel="Ronda", ylabel="Pérdida logarítmica", title="Cuándo paramos")
ax.legend()
ConfusionMatrixDisplay.from_predictions(y[validacion], pred_val, labels=np.arange(len(clases)),
                                        display_labels=clases, ax=cm, colorbar=False)
cm.set(xlabel="Predicción", ylabel="Etiqueta", title="Segmentos de validación")
fig.tight_layout()
plt.show()

Lee la curva de validación alrededor de la ronda elegida. Al predecir usamos `iteration_range=(0, rondas)` para aplicar solo las rondas seleccionadas.

<span style="color:gold">**Una segunda imagen**</span>

Mostraremos el resultado en `skarn_2`, fuera del ajuste y de la elección de rondas. Las dos vistas se guardaron con el mismo muestreo de filas y columnas. Comprueba visualmente que sus estructuras coincidan antes de interpretar el mapa. La segunda imagen no trae etiquetas independientes para calcular exactitud.

In [ ]:
with np.load("files/xgboost/skarn_2_vistas.npz") as archivo:
    paralelos, cruzados = archivo["np"], archivo["nc"]
assert paralelos.shape == cruzados.shape
alto, ancho, _ = paralelos.shape
seis_canales = np.concatenate([paralelos, cruzados], axis=2)
X_imagen = seis_canales.reshape(-1, 6)
dimg = xgb.DMatrix(X_imagen, feature_names=columnas)
proba_img = model.predict(dimg, iteration_range=(0, rondas))
mapa = proba_img.argmax(axis=1).reshape(alto, ancho)

fig, ejes = plt.subplots(1, 3, figsize=(11, 4))
for eje, imagen, nombre in zip(ejes[:2], [paralelos, cruzados], ["NP", "NC"]):
    eje.imshow(imagen)
    eje.set_title(nombre)
im = ejes[2].imshow(mapa, cmap=ListedColormap(["#E69F00", "#7B3294", "#009E73"]),
                    vmin=-0.5, vmax=len(clases)-0.5, interpolation="nearest")
ejes[2].set_title("Anotación predicha")
fig.colorbar(im, ax=ejes[2], ticks=np.arange(len(clases))).ax.set_yticklabels(clases)
for eje in ejes:
    eje.axis("off")
plt.show()

<a id="parte-4"></a>

### <span style="color:lightgreen">**En conclusión...**</span>

XGBoost suma correcciones en varias rondas. Aquí leímos los canales de dos vistas para una misma posición, separamos segmentos completos para validación y dibujamos las anotaciones predichas en otra imagen.

**Comprueba tu lectura.** Explica qué error introduciría emparejar un píxel NP con otro NC de distinta coordenada. Diseña una prueba con nuevas láminas anotadas y conserva sus identificadores desde la lectura de los datos.

---